### Import libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

import imblearn

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import(accuracy_score, precision_score, recall_score, f1_score, 
                            roc_auc_score, confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier

### Load Dataset

In [2]:
file_path = r"C:\Users\HP\Desktop\Infotact Internship\week2_fused_dataset.csv"

In [3]:
df = pd.read_csv(file_path)

### Dataset Information

In [4]:
df.shape

(10000, 39)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 39 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   UDI                                   10000 non-null  int64  
 1   Product ID                            10000 non-null  str    
 2   Air temperature [K]                   10000 non-null  float64
 3   Process temperature [K]               10000 non-null  float64
 4   Rotational speed [rpm]                10000 non-null  int64  
 5   Torque [Nm]                           10000 non-null  float64
 6   Tool wear [min]                       10000 non-null  int64  
 7   Machine failure                       10000 non-null  int64  
 8   TWF                                   10000 non-null  int64  
 9   HDF                                   10000 non-null  int64  
 10  PWF                                   10000 non-null  int64  
 11  OSF                        

### Check for Missing Values

In [6]:
df.isnull().sum()

UDI                                     0
Product ID                              0
Air temperature [K]                     0
Process temperature [K]                 0
Rotational speed [rpm]                  0
Torque [Nm]                             0
Tool wear [min]                         0
Machine failure                         0
TWF                                     0
HDF                                     0
PWF                                     0
OSF                                     0
RNF                                     0
Air temperature [K]_rolling_mean        0
Air temperature [K]_rolling_std         0
Air temperature [K]_rolling_var         0
Process temperature [K]_rolling_mean    0
Process temperature [K]_rolling_std     0
Process temperature [K]_rolling_var     0
Rotational speed [rpm]_rolling_mean     0
Rotational speed [rpm]_rolling_std      0
Rotational speed [rpm]_rolling_var      0
Torque [Nm]_rolling_mean                0
Torque [Nm]_rolling_std           

The above output shows there is no missing values.

### Check Class Distribution

In [7]:
target = 'Machine failure'

df[target].value_counts()

Machine failure
0    9661
1     339
Name: count, dtype: int64

Thus the target have a imbalanced dataset, which is needed to be treated else it cannot perform good prediction.

### Prepare Features and Target

In [8]:
X = df.drop(columns=[target])

#### checking do we have a string, as string not accepted in SMOTE

In [9]:
print(X.select_dtypes(include=['object', 'string']).columns)

Index(['Product ID', 'Timestamp'], dtype='str')


In [10]:
# Remove identifier columns
X = X.drop(columns=["Product ID", "Timestamp"])

In [11]:
X.dtypes

UDI                                       int64
Air temperature [K]                     float64
Process temperature [K]                 float64
Rotational speed [rpm]                    int64
Torque [Nm]                             float64
Tool wear [min]                           int64
TWF                                       int64
HDF                                       int64
PWF                                       int64
OSF                                       int64
RNF                                       int64
Air temperature [K]_rolling_mean        float64
Air temperature [K]_rolling_std         float64
Air temperature [K]_rolling_var         float64
Process temperature [K]_rolling_mean    float64
Process temperature [K]_rolling_std     float64
Process temperature [K]_rolling_var     float64
Rotational speed [rpm]_rolling_mean     float64
Rotational speed [rpm]_rolling_std      float64
Rotational speed [rpm]_rolling_var      float64
Torque [Nm]_rolling_mean                

#### checking do we have any special characters in column names, because thats not accepted by LGBM

In [12]:
for col in X.columns:
    if any(ch in str(col) for ch in ['"', "'", "[", "]", "{", "}", ":", ","]):
        print(col)

Air temperature [K]
Process temperature [K]
Rotational speed [rpm]
Torque [Nm]
Tool wear [min]
Air temperature [K]_rolling_mean
Air temperature [K]_rolling_std
Air temperature [K]_rolling_var
Process temperature [K]_rolling_mean
Process temperature [K]_rolling_std
Process temperature [K]_rolling_var
Rotational speed [rpm]_rolling_mean
Rotational speed [rpm]_rolling_std
Rotational speed [rpm]_rolling_var
Torque [Nm]_rolling_mean
Torque [Nm]_rolling_std
Torque [Nm]_rolling_var
Tool wear [min]_rolling_mean
Tool wear [min]_rolling_std
Tool wear [min]_rolling_var


LightGBM wont accept these names with the special characters, which has to be treated befor modelling.

#### clean the column names

In [13]:
import re

X.columns = [
    re.sub(r'[^A-Za-z0-9_]+', '_', str(col))
    for col in X.columns
]

In [14]:
X.shape

(10000, 36)

In [15]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 36 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   UDI                                  10000 non-null  int64  
 1   Air_temperature_K_                   10000 non-null  float64
 2   Process_temperature_K_               10000 non-null  float64
 3   Rotational_speed_rpm_                10000 non-null  int64  
 4   Torque_Nm_                           10000 non-null  float64
 5   Tool_wear_min_                       10000 non-null  int64  
 6   TWF                                  10000 non-null  int64  
 7   HDF                                  10000 non-null  int64  
 8   PWF                                  10000 non-null  int64  
 9   OSF                                  10000 non-null  int64  
 10  RNF                                  10000 non-null  int64  
 11  Air_temperature_K__rolling_mean      100

In [16]:
y = df[target]

### Stratified 5-Fold Cross Validation

In [17]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [18]:
skf.split(X, y)

<generator object _BaseKFold.split at 0x0000022F94924700>

### Lists to Store Metrics

In [19]:
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
auc_scores = []

### Training Loop

In [20]:
fold = 1

for train_index, test_index in skf.split(X, y):
    print("="*60)
    print(f"Fold {fold}")

    x_train, x_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    smote = SMOTE(random_state=42)

    x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

    print("Original Training shape:", y_train.value_counts().to_dict())
    print("Resampled Training shape:", y_train_smote.value_counts().to_dict())

    # modelling
    model = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42)

    model.fit(x_train_smote, y_train_smote)

    y_pred = model.predict(x_test)

    y_prob = model.predict_proba(x_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    auc_scores.append(auc)

    print("\n classification_report \n")
    print(classification_report(y_test, y_pred))
    print("\n confusion_matrix \n")
    print(confusion_matrix(y_test, y_pred))
    print(f"AUC : {auc:.4f}")

    fold += 1


Fold 1
Original Training shape: {0: 7728, 1: 272}
Resampled Training shape: {0: 7728, 1: 7728}
[LightGBM] [Info] Number of positive: 7728, number of negative: 7728
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010336 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6994
[LightGBM] [Info] Number of data points in the train set: 15456, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000

 classification_report 

              precision    recall  f1-score   support

           0       1.00      0.99      1.00      1933
           1       0.82      0.97      0.89        67

    accuracy                           0.99      2000
   macro avg       0.91      0.98      0.94      2000
weighted avg       0.99      0.99      0.99      2000


 confusion_matrix 

[[1919   14]
 [   2   65]]
AUC : 0.9921
Fold 2
Original Training shape: {0: 7729, 1: 271}
Resa

### Understanding from Metrics

1) Accuracy = 99%, Excellent, but accuracy alone isn't the best metric for imbalanced data.
2) Precision (Failure class) = 82%, model predicts the failure.
3) Recall (Failure class) = 97%, The model detects 97% of actual failures. This is very good for predictive maintenance.
4) F1-score = 89%, Strong balance between precision and recall.
5) ROC-AUC = 99%, Outstanding class separation.

CONFUSION MATRIX

1) 1924 normal machines correctly identified.
2) 67 failures correctly detected.
3) 8 false alarms (normal predicted as failure).
4) 1 missed failure.

### Average Results

In [21]:
print("="*70)

print("Average Accuracy :", np.mean(accuracy_scores))
print("Average Precision :", np.mean(precision_scores))
print("Average Recall :", np.mean(recall_scores))
print("Average F1 Score :", np.mean(f1_scores))
print("Average ROC-AUC :", np.mean(auc_scores))

Average Accuracy : 0.9936999999999999
Average Precision : 0.8623788798335357
Average Recall : 0.9705004389815628
Average F1 Score : 0.9129253819431611
Average ROC-AUC : 0.9894565602095365


### Feature Importance

In [23]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=True
)

importance.head(20)

,Feature,Importance
10,RNF,0
11,Air_temperature_K__rolling_mean,104
25,Tool_wear_min__rolling_var,107
22,Torque_Nm__rolling_var,112
19,Rotational_speed_rpm__rolling_var,116
24,Tool_wear_min__rolling_std,133
8,PWF,139
9,OSF,145
23,Tool_wear_min__rolling_mean,146
7,HDF,149
